## Qwen2.5-VL-7B BF16 smoke test — Train Validation only

이 Notebook은 `Qwen/Qwen2.5-VL-7B-Instruct`를 BF16으로 로드하고, Prompt v2를 고정된 Train Validation 50개에서만 평가합니다. 모델 학습과 Test 접근은 하지 않습니다.

목적은 7B 모델의 GPU 호환성, VRAM, 추론 속도, parsing, Exact Match와 prediction collapse를 확인하는 것입니다.

### 1. GPU와 BF16 확인

In [ ]:
import torch

assert torch.cuda.is_available(), 'GPU runtime이 아닙니다.'
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert torch.cuda.is_bf16_supported(), 'BF16을 지원하지 않는 GPU입니다.'
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'GPU: {gpu_name}')
print(f'VRAM: {gpu_memory_gb:.1f} GB')
print(f'BF16 supported: {torch.cuda.is_bf16_supported()}')

### 2. Google Drive 프로젝트 연결

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/SNU')

required_paths = [
    PROJECT_DIR / 'data' / 'train.csv',
    PROJECT_DIR / 'data' / 'train',
    PROJECT_DIR / 'outputs' / 'splits' / 'validation_ids.csv',
    PROJECT_DIR / 'src' / 'qwen_baseline.py',
    PROJECT_DIR / 'scripts' / 'evaluate_qwen_baseline.py',
]
for path in required_paths:
    assert path.exists(), f'필수 경로를 찾지 못했습니다: {path}'
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print(f'Project: {PROJECT_DIR}')

### 3. 실행 패키지 설치

Colab의 CUDA PyTorch는 교체하지 않고 Qwen 실행 패키지만 설치합니다.

In [ ]:
%pip install -q transformers==5.13.0 accelerate==1.14.0 qwen-vl-utils==0.0.14

### 4. Prompt와 Answer 변환 테스트

16GB 이상 모델을 받기 전에 Prompt v2와 direct Answer parser 테스트를 실행합니다.

In [ ]:
import subprocess

test_file = PROJECT_DIR / 'tests' / 'test_qwen_baseline.py'
assert test_file.is_file(), f'테스트 파일을 찾지 못했습니다: {test_file}'
subprocess.run(
    [
        sys.executable, '-m', 'unittest', 'discover',
        '-s', str(PROJECT_DIR / 'tests'),
        '-p', 'test_qwen_baseline.py',
        '-v',
    ],
    cwd=PROJECT_DIR,
    check=True,
)

### 5. Qwen2.5-VL-7B BF16 Validation 50개 smoke test

최초 실행에는 약 16.6GB 모델 다운로드가 필요합니다. seed 42의 동일한 50개를 사용하며 결과는 2B 출력과 다른 폴더에 저장합니다.

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-VL-7B-Instruct'
CACHE_DIR = Path('/content/huggingface/hub')
OUTPUT_BASE = PROJECT_DIR / 'outputs' / 'qwen25_vl_7b_prompt_v2'

smoke_command = [
    sys.executable, '-m', 'scripts.evaluate_qwen_baseline',
    '--data-dir', str(PROJECT_DIR / 'data'),
    '--validation-split-csv', str(PROJECT_DIR / 'outputs' / 'splits' / 'validation_ids.csv'),
    '--output-dir', str(OUTPUT_BASE),
    '--model-name', MODEL_NAME,
    '--prompt-version', 'v2_direct_answer',
    '--cache-dir', str(CACHE_DIR),
    '--device', 'cuda',
    '--dtype', 'bfloat16',
    '--limit', '50',
    '--seed', '42',
    '--max-new-tokens', '16',
]
subprocess.run(smoke_command, check=True)

### 6. 결과 확인

Exact Match뿐 아니라 identity 쏠림과 비-identity 정답 복원 여부를 확인합니다.

In [ ]:
import json
import pandas as pd
from IPython.display import display

summary_path = OUTPUT_BASE / 'validation_50' / 'summary.json'
predictions_path = OUTPUT_BASE / 'validation_50' / 'predictions.csv'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
predictions = pd.read_csv(predictions_path)
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(predictions.head(10))

identity = '[1, 2, 3, 4]'
ordered = predictions[predictions['target_answer'] != identity]
ordered_exact = (ordered['target_answer'] == ordered['predicted_answer']).mean()
print(f'비-identity 샘플 Exact Match: {ordered_exact:.4f} ({(ordered.target_answer == ordered.predicted_answer).sum()}/{len(ordered)})')